# Coleção de Dataset de Landmarks (Gestos)
Este notebook é derivado do reconhecimento de gestos em tempo real. A diferença é que o objetivo deste notebook é gravar a posição x, y, z de cada um dos 21 pontos (landmarks) da sua mão detectada, anexá-los a um 'label' (nome do gesto) específico definido por você, e exportar tudo rotulado para um arquivo `.csv`.

In [16]:
# Instalando as dependências necessárias
!pip install opencv-python mediapipe pandas


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
import urllib.request
import os

# O MediaPipe requer o modelo treinado para gerar os landmarks.
model_path = 'gesture_recognizer.task'
if not os.path.exists(model_path):
    print("Baixando o modelo de gestos...")
    url = 'https://storage.googleapis.com/mediapipe-models/gesture_recognizer/gesture_recognizer/float16/1/gesture_recognizer.task'
    urllib.request.urlretrieve(url, model_path)
    print("Modelo baixado com sucesso!")
else:
    print("O modelo já existe localmente.")

O modelo já existe localmente.


In [18]:
import cv2
import mediapipe as mp
import csv
import time

# ======= CONFIGURAÇÕES DO DATASET =======
DATASET_LABEL = "Heart" # <- MUDE AQUI O NOME DO GESTO QUE VOCE ESTA GRAVANDO
CSV_FILE_NAME = "hand_landmarks_dataset.csv"

print(f"Os frames capturados vão ser salvos com a label alvo: '{DATASET_LABEL}'")

Os frames capturados vão ser salvos com a label alvo: 'Heart'


In [19]:
# Inicializando os módulos correspondentes do MediaPipe
BaseOptions = mp.tasks.BaseOptions
GestureRecognizer = mp.tasks.vision.GestureRecognizer
GestureRecognizerOptions = mp.tasks.vision.GestureRecognizerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

# Configurando as opções para o reconhecedor
options = GestureRecognizerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.IMAGE, # Modo de imagem
    num_hands=1 # Para construir um bom dataset é melhor usar 1 mão de cada vez
)
recognizer = GestureRecognizer.create_from_options(options)

# Inicializa o header (cabeçalho) do arquivo CSV, criando as colunas de x0,y0,z0 até x20,y20,z20 e o 'target'
def build_csv_header():
    header = ['target']
    for i in range(21):
        header.extend([f'x{i}', f'y{i}', f'z{i}'])
    return header

# Criação do arquivo caso não exista e inserção do cabeçalho
if not os.path.exists(CSV_FILE_NAME):
    with open(CSV_FILE_NAME, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(build_csv_header())
    print(f"Arquivo '{CSV_FILE_NAME}' criado com o cabecalho.")
else:
    print(f"Arquivo '{CSV_FILE_NAME}' já existe. As novas leituras serão adicionadas como linhas nele!")

Arquivo 'hand_landmarks_dataset.csv' já existe. As novas leituras serão adicionadas como linhas nele!


In [20]:
# ====== INICIANDO GRAVAÇÃO DO DATASET ========
mp_hands = mp.tasks.vision.HandLandmarksConnections
mp_drawing = mp.tasks.vision.drawing_utils
mp_drawing_styles = mp.tasks.vision.drawing_styles
cap = cv2.VideoCapture(0)

frames_saved = 0 
print(f"Preparado para gravar posicoes para '{DATASET_LABEL}'.")
print("PRESSIONE 's' para salvar as coordenadas atuais no CSV.")
print("PRESSIONE 'q' para sair e parar a gravação.")

try:
    while cap.isOpened():
        success, image = cap.read()
        if not success:
            print("Captura da webcam falhou ou arquivo terminou. Saindo...")
            break

        # Transformando RGB para passar no mediapipe
        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)

        # Extrai os landmarks via recognition
        recognition_result = recognizer.recognize(mp_image)
        
        # Variaveis para gravar no CSV
        current_landmark_row = []

        # Vamos pegar sempre da primeira mão que aparecer no quadro
        if recognition_result.hand_landmarks:
            primeira_mao = recognition_result.hand_landmarks[0]
            
            # Formatando a array: a Label primeiro, depois x,y,z dos 21 pontos
            current_landmark_row = [DATASET_LABEL]
            for landmark in primeira_mao:
                current_landmark_row.extend([landmark.x, landmark.y, landmark.z])
            
            # Desenhar na tela pra se guiar (Opcional, mas legal de ver o que tá pegando)
            mp_drawing.draw_landmarks(
                image,
                primeira_mao,
                mp_hands.HAND_CONNECTIONS,
                mp_drawing_styles.get_default_hand_landmarks_style(),
                mp_drawing_styles.get_default_hand_connections_style()
            )

        text_info = f"{DATASET_LABEL} | Salvos: {frames_saved} | Aperte 'S' p/ Gravar"
        cv2.putText(image, text_info, (10, 30), cv2.FONT_HERSHEY_DUPLEX, 0.7, (255, 0, 0), 1)

        cv2.imshow('Gravar Dataset - CSV', image)
        
        key = cv2.waitKey(1) & 0xFF
        # Sai se apertou Q
        if key == ord('q'):
            break
        # Salva o frame se apertou S enquanto havia mão visível
        elif key == ord('s') and len(current_landmark_row) > 0:
            with open(CSV_FILE_NAME, mode='a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(current_landmark_row)
            frames_saved += 1
            print(f"Coordenada ({DATASET_LABEL}) gravada no CSV!")

except KeyboardInterrupt:
    print("Script interrompido manual.")
finally:
    cap.release()
    cv2.destroyAllWindows()
    print(f"Programa encerrado. Total de posicoes '{DATASET_LABEL}' salvas na sessao: {frames_saved}")
    print(f"Tudo exportado para o arquivo {CSV_FILE_NAME}")

Preparado para gravar posicoes para 'Heart'.
PRESSIONE 's' para salvar as coordenadas atuais no CSV.
PRESSIONE 'q' para sair e parar a gravação.
Coordenada (Heart) gravada no CSV!
Coordenada (Heart) gravada no CSV!
Coordenada (Heart) gravada no CSV!
Coordenada (Heart) gravada no CSV!
Coordenada (Heart) gravada no CSV!
Coordenada (Heart) gravada no CSV!
Coordenada (Heart) gravada no CSV!
Coordenada (Heart) gravada no CSV!
Coordenada (Heart) gravada no CSV!
Coordenada (Heart) gravada no CSV!
Coordenada (Heart) gravada no CSV!
Coordenada (Heart) gravada no CSV!
Coordenada (Heart) gravada no CSV!
Coordenada (Heart) gravada no CSV!
Coordenada (Heart) gravada no CSV!
Coordenada (Heart) gravada no CSV!
Coordenada (Heart) gravada no CSV!
Coordenada (Heart) gravada no CSV!
Coordenada (Heart) gravada no CSV!
Coordenada (Heart) gravada no CSV!
Coordenada (Heart) gravada no CSV!
Coordenada (Heart) gravada no CSV!
Coordenada (Heart) gravada no CSV!
Coordenada (Heart) gravada no CSV!
Coordenada (Hea